In [1]:
import pandas as pd
products = pd.read_csv('../data/raw/products.csv')
print(products.head())


  product_id             product_name       category machine_type  \
0       P001    Hydraulic Pump HP-300      Hydraulic    Excavator   
1       P002  Engine Oil Filter OF-90        Filters       Loader   
2       P003        Air Filter AF-120        Filters    Excavator   
3       P004        Track Chain TC-45  Undercarriage    Bulldozer   
4       P005     Fuel Injector FI-220         Engine    Excavator   

          brand  model_compatibility  unit_cost  unit_price  \
0           CAT             CAT 320D      18950       24500   
1           JCB              JCB 3DX        450         720   
2       Komatsu        Komatsu PC210        820        1250   
3         Volvo             Volvo D6      32500       39800   
4  Tata Hitachi  Tata Hitachi EX 200       6350        8450   

   margin_percentage  gst_rate  ...  warranty_months reorder_level  \
0               29.2        28  ...               12            15   
1               37.3        18  ...                6            80

In [2]:
print("=== INFO ===")
print(products.info())

print("\n=== MISSING VALUES ===")
print(products.isnull().sum())

print("\n=== SAMPLE DATA ===")
products.head()

=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           30 non-null     str    
 1   product_name         30 non-null     str    
 2   category             30 non-null     str    
 3   machine_type         30 non-null     str    
 4   brand                30 non-null     str    
 5   model_compatibility  30 non-null     str    
 6   unit_cost            30 non-null     int64  
 7   unit_price           30 non-null     int64  
 8   margin_percentage    30 non-null     float64
 9   gst_rate             30 non-null     int64  
 10  weight_kg            30 non-null     float64
 11  dimensions_cm        30 non-null     str    
 12  material_type        30 non-null     str    
 13  warranty_months      30 non-null     int64  
 14  reorder_level        30 non-null     int64  
 15  safety_stock         30 non-null     int

,product_id,product_name,category,machine_type,brand,model_compatibility,unit_cost,unit_price,margin_percentage,gst_rate,...,warranty_months,reorder_level,safety_stock,max_stock_level,lead_time_days,criticality_level,usage_frequency,uom,last_purchase_price,last_purchase_date
0,P001,Hydraulic Pump HP-300,Hydraulic,Excavator,CAT,CAT 320D,18950,24500,29.2,28,...,12,15,10,40,21,High,Medium,piece,18500,2019-03-15
1,P002,Engine Oil Filter OF-90,Filters,Loader,JCB,JCB 3DX,450,720,37.3,18,...,6,80,40,200,7,Medium,High,piece,430,2019-07-22
2,P003,Air Filter AF-120,Filters,Excavator,Komatsu,Komatsu PC210,820,1250,52.4,18,...,6,60,30,150,7,Medium,High,piece,800,2020-01-10
3,P004,Track Chain TC-45,Undercarriage,Bulldozer,Volvo,Volvo D6,32500,39800,22.5,28,...,18,8,4,20,30,High,Low,set,32000,2020-06-18
4,P005,Fuel Injector FI-220,Engine,Excavator,Tata Hitachi,Tata Hitachi EX 200,6350,8450,33.0,28,...,12,30,15,80,18,High,Medium,piece,6200,2020-11-05


In [3]:
text_cols = [
    'product_id', 'product_name', 'category', 'machine_type',
    'brand', 'model_compatibility', 'dimensions_cm',
    'material_type', 'criticality_level'
]

for col in text_cols:
    products[col] = products[col].str.title()


In [4]:
dims = products['dimensions_cm'].str.lower().str.split('x', expand=True)
products['length_cm'] = pd.to_numeric(dims[0], errors='coerce')
products['width_cm'] = pd.to_numeric(dims[1], errors='coerce')
products['height_cm'] = pd.to_numeric(dims[2], errors='coerce')


In [10]:
products['margin_amount'] = (
    products['unit_price'] - products['unit_cost']
)

products['gst_amount'] = (
    products['unit_price'] * (products['gst_rate'] / 100)
)

products['landed_price'] = products['unit_price'] + products['gst_amount']


products['volume_cm3'] = (
    products['length_cm'] * products['width_cm'] * products['height_cm']
)


products['stock_buffer_pct'] = (
    products['safety_stock'] / products['max_stock_level']
)

products['lead_time_risk_flag'] = (
    products['lead_time_days'] > products['reorder_level']
).astype(int)


In [11]:
products.drop_duplicates(inplace=True)


In [12]:
print("=== DUPLICATE PRODUCT IDs ===")
print(products['product_id'].duplicated().sum())

print("\n=== CATEGORY VALUES ===")
print(products['category'].unique())

print("\n=== CRITICALITY LEVELS ===")
print(products['criticality_level'].unique())

print("\n=== NUMERIC SUMMARY ===")
numeric_cols = [
    'unit_cost', 'unit_price', 'margin_percentage',
    'weight_kg', 'warranty_months', 'reorder_level',
    'safety_stock', 'max_stock_level', 'lead_time_days'
]
print(products[numeric_cols].describe())


=== DUPLICATE PRODUCT IDs ===
0

=== CATEGORY VALUES ===
<StringArray>
[    'Hydraulic',       'Filters', 'Undercarriage',        'Engine',
    'Electrical',   'Attachments',        'Brakes',         'Seals',
  'Transmission',       'Cooling',         'Belts', 'Pins & Bushes',
         'Cabin',    'Lubricants',       'Sensors']
Length: 15, dtype: str

=== CRITICALITY LEVELS ===
<StringArray>
['High', 'Medium', 'Low']
Length: 3, dtype: str

=== NUMERIC SUMMARY ===
          unit_cost     unit_price  margin_percentage   weight_kg  \
count     30.000000      30.000000          30.000000   30.000000   
mean   14585.000000   18593.666667          40.666667   36.933333   
std    21478.824014   26542.408160          16.062149   79.311155   
min      180.000000     310.000000          20.800000    0.200000   
25%      957.500000    1450.000000          29.575000    0.825000   
50%     7600.000000   10250.000000          36.700000    7.900000   
75%    18312.500000   23850.000000          51.30

In [13]:
products.to_csv('../data/processed/products.csv', index=False)
print("Processed products.csv saved to data/processed/")


Processed products.csv saved to data/processed/
